# 01 — Auditoría inicial de datos

**Proyecto EPBI — Economic Perception Bias in Europe**

> **Pipeline:** **01 Auditoría** → 02 Limpieza ESS → 03 Eurostat → 04 Construcción EPBI → 05 Integración micro–macro → 06 Econometría → 07 Datos para dashboard

Este notebook abre el flujo de trabajo. Su función es **conocer y documentar los microdatos originales de la European Social Survey (ESS) antes de realizar cualquier transformación**. La auditoría establece qué variables están disponibles, con qué cobertura y qué incidencias deberán resolverse en la fase de limpieza.

La regla de esta etapa es sencilla: **observar y diagnosticar, no transformar**. Los datos brutos permanecen intactos; las decisiones derivadas de la auditoría se aplicarán en el notebook 02.

## Objetivos de la etapa

- localizar y cargar el fichero ESS original;
- comprobar dimensiones, tipos de datos, duplicados y uso de memoria;
- analizar valores ausentes y códigos especiales;
- revisar las variables nucleares del proyecto;
- medir la cobertura por **ronda** y por **país-ronda**;
- identificar combinaciones en las que no puede construirse el EPBI;
- buscar variables históricas o alternativas relacionadas con la renta;
- exportar un informe de auditoría reproducible que sirva de base para la limpieza.

Al finalizar, esta fase no genera una nueva muestra analítica: genera **evidencia y criterios de decisión** para el notebook 02.


## 1. Librerías y configuración

Se prepara el entorno de trabajo utilizado durante la auditoría: librerías, opciones de visualización y parámetros generales. En esta fase todavía no se modifica ningún dato.


In [ ]:
from pathlib import Path
from datetime import datetime
import warnings
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 160)
pd.set_option("display.max_rows", 120)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", "{:,.2f}".format)

print("Fecha de ejecución:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))


## 2. Rutas del proyecto

Con el entorno preparado, se definen de forma reproducible las carpetas de entrada y salida. Esto permite separar claramente los datos brutos, los datos procesados, los gráficos y los informes generados por el proyecto.


In [ ]:
CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name.upper() == "CODIGO" else CURRENT_DIR

CODIGO = PROJECT_ROOT / "CODIGO"
DATOS = PROJECT_ROOT / "DATOS"
DATOS_BRUTOS = DATOS / "BRUTOS"
DATOS_PROCESADOS = DATOS / "PROCESADOS"
GRAFICOS = PROJECT_ROOT / "GRAFICOS"
INFORMES = PROJECT_ROOT / "INFORMES"

for carpeta in [CODIGO, DATOS, DATOS_BRUTOS, DATOS_PROCESADOS, GRAFICOS, INFORMES]:
    carpeta.mkdir(parents=True, exist_ok=True)

print("Raíz del proyecto:", PROJECT_ROOT)
print("Datos brutos:", DATOS_BRUTOS)
print("Informes:", INFORMES)


## 3. Funciones auxiliares

Antes de inspeccionar los microdatos se definen pequeñas funciones reutilizables para resumir variables, clasificar coberturas y organizar los diagnósticos. Centralizar estas operaciones evita repetir lógica en las secciones posteriores.


In [ ]:
def listar_archivos(carpeta: Path) -> pd.DataFrame:
    registros = []
    for elemento in carpeta.iterdir():
        registros.append({
            "nombre": elemento.name,
            "tipo": "carpeta" if elemento.is_dir() else "archivo",
            "ruta": str(elemento.resolve()),
            "extension": "".join(elemento.suffixes) if elemento.is_file() else "",
            "tamano_mb": round(elemento.stat().st_size / (1024**2), 2) if elemento.is_file() else np.nan,
            "fecha_modificacion": datetime.fromtimestamp(elemento.stat().st_mtime).strftime("%Y-%m-%d %H:%M:%S"),
        })
    if not registros:
        return pd.DataFrame(columns=["nombre","tipo","ruta","extension","tamano_mb","fecha_modificacion"])
    return pd.DataFrame(registros).sort_values(["tipo","nombre"]).reset_index(drop=True)


def audit_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    memoria = df.memory_usage(deep=True).iloc[1:] / (1024**2)
    out = pd.DataFrame({
        "variable": df.columns,
        "tipo_dato": df.dtypes.astype(str).values,
        "no_nulos": df.notna().sum().values,
        "valores_perdidos": df.isna().sum().values,
        "porcentaje_perdidos": (df.isna().mean() * 100).round(2).values,
        "valores_unicos": df.nunique(dropna=True).values,
        "memoria_mb": memoria.round(4).values,
    })
    out["es_constante"] = out["valores_unicos"] <= 1
    out["completamente_vacia"] = out["no_nulos"] == 0
    out["posible_id_unico"] = out["valores_unicos"] == len(df)
    return out


def resumen_general(df: pd.DataFrame, nombre: str) -> pd.DataFrame:
    return pd.DataFrame({
        "metrica": [
            "nombre_dataset","numero_filas","numero_columnas",
            "duplicados_completos","memoria_total_mb","fecha_auditoria"
        ],
        "valor": [
            nombre, len(df), df.shape[1], int(df.duplicated().sum()),
            round(df.memory_usage(deep=True).sum()/(1024**2), 2),
            datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        ]
    })


def estado_cobertura(pct):
    if pd.isna(pct):
        return "Sin evaluar"
    if pct == 0:
        return "Sin datos"
    if pct < 50:
        return "Cobertura muy baja"
    if pct < 75:
        return "Cobertura baja"
    if pct < 90:
        return "Cobertura media"
    return "Cobertura alta"


def tabla_frecuencias(df, variable, max_valores=30):
    if variable not in df.columns:
        return pd.DataFrame()
    s = df[variable]
    t = s.value_counts(dropna=False).head(max_valores).rename_axis("valor").reset_index(name="casos")
    t["porcentaje"] = (100 * t["casos"] / len(df)).round(2)
    t.insert(0, "variable", variable)
    return t


## 4. Inventario de datos brutos

El primer diagnóstico consiste en comprobar qué ficheros existen realmente en la carpeta de datos brutos. Este inventario permite verificar la materia prima disponible antes de seleccionar el fichero ESS que se utilizará en el resto del pipeline.


In [ ]:
inventario_brutos = listar_archivos(DATOS_BRUTOS)
display(inventario_brutos)


## 5. Localización y carga de los microdatos ESS

A partir del inventario anterior se localiza el fichero ESS original y se carga sin transformaciones. Desde este punto, todas las comprobaciones se realizan sobre una copia en memoria de los microdatos brutos.


In [ ]:
# Se aceptan CSV, CSV.GZ y XLSX. Se priorizan nombres que contengan 'microdatos'.
candidatos = []
for patron in ["*microdatos*.csv", "*microdatos*.csv.gz", "*microdatos*.xlsx", "*ESS*.csv", "*ESS*.xlsx"]:
    candidatos.extend(DATOS_BRUTOS.glob(patron))

# Quitar duplicados manteniendo orden.
vistos = set()
candidatos = [p for p in candidatos if not (str(p.resolve()) in vistos or vistos.add(str(p.resolve())))]

if not candidatos:
    raise FileNotFoundError(f"No se encontraron microdatos ESS en {DATOS_BRUTOS}")

if len(candidatos) > 1:
    print("Candidatos encontrados:")
    for i, p in enumerate(candidatos, start=1):
        print(i, p.name)

# Preferencia por ESS11_microdatos.csv/csv.gz/xlsx si existe.
preferidos = [
    DATOS_BRUTOS / "ESS11_microdatos.csv",
    DATOS_BRUTOS / "ESS11_microdatos.csv.gz",
    DATOS_BRUTOS / "ESS11_microdatos.xlsx",
]
ess_file = next((p for p in preferidos if p.exists()), candidatos[0])

print("Archivo seleccionado:", ess_file.name)
print("Tamaño MB:", round(ess_file.stat().st_size/(1024**2), 2))

nombre = ess_file.name.lower()
if nombre.endswith(".csv.gz"):
    ess_raw = pd.read_csv(ess_file, compression="gzip", low_memory=False, encoding="utf-8-sig")
elif nombre.endswith(".csv"):
    ess_raw = pd.read_csv(ess_file, low_memory=False, encoding="utf-8-sig")
elif nombre.endswith(".xlsx"):
    ess_raw = pd.read_excel(ess_file, engine="openpyxl")
else:
    raise ValueError(f"Formato no soportado: {ess_file.suffix}")

print("Dimensiones:", ess_raw.shape)
display(ess_raw.head())


## 6. Auditoría general

Una vez cargada la ESS, se obtiene una primera visión global de su estructura: número de observaciones y variables, tipos de datos, valores ausentes, cardinalidad y posibles duplicados completos. Este resumen sirve como referencia para interpretar los diagnósticos más específicos que siguen.


In [ ]:
resumen_ess = resumen_general(ess_raw, "ESS microdatos originales")
audit_ess = audit_dataframe(ess_raw)

tipos_ess = (
    ess_raw.dtypes.astype(str)
    .value_counts()
    .rename_axis("tipo_dato")
    .reset_index(name="numero_columnas")
)

duplicados_ess = pd.DataFrame({
    "metrica": ["filas_totales","duplicados_completos","porcentaje_duplicados"],
    "valor": [
        len(ess_raw),
        int(ess_raw.duplicated().sum()),
        round(100 * ess_raw.duplicated().mean(), 4),
    ]
})

display(resumen_ess)
display(tipos_ess)
display(duplicados_ess)
display(audit_ess.sort_values("porcentaje_perdidos", ascending=False).head(40))


## 7. Variables nucleares del proyecto

Después de la revisión general, la auditoría se centra en las variables necesarias para construir el EPBI y estimar los modelos posteriores. Se comprueba su presencia y disponibilidad antes de tomar cualquier decisión de limpieza.


In [ ]:
VARIABLES_NUCLEARES = [
    # Identificación
    "idno","cntry","essround","edition","proddate",
    # Pesos
    "dweight","pspwght","pweight","anweight",
    # EPBI
    "hinctnta","hincfel",
    # Individuales
    "agea","gndr","eisced","eduyrs","mnactic","hhmmb",
    # Actitudinales
    "lrscale","gincdif","ppltrst","trstprl","trstplt",
    "stfdem","stfeco","stflife",
]

comprobacion_variables = pd.DataFrame({
    "variable": VARIABLES_NUCLEARES,
    "disponible": [v in ess_raw.columns for v in VARIABLES_NUCLEARES]
})

if "essround" in ess_raw.columns:
    rondas_con_datos = []
    for v in VARIABLES_NUCLEARES:
        if v in ess_raw.columns:
            n = ess_raw.groupby("essround")[v].count()
            rondas_con_datos.append(int((n > 0).sum()))
        else:
            rondas_con_datos.append(0)
    comprobacion_variables["rondas_con_algun_dato"] = rondas_con_datos

display(comprobacion_variables)


## 8. Búsqueda de variables relacionadas o alternativas

La disponibilidad de algunas variables cambia entre rondas. Por ello, se buscan nombres relacionados o posibles alternativas históricas, especialmente en las variables de renta, con el objetivo de documentar qué opciones existen antes de fijar la cobertura temporal del análisis.


In [ ]:
# Muy útil para detectar variables históricas o equivalentes que no estén en la lista inicial.
PATRONES = {
    "renta": r"hinc|income",
    "educacion": r"educ|eisced|eduyrs",
    "peso": r"weight|wght",
    "actividad": r"mnactic|activity|mainact",
    "genero": r"gndr|gender|sex",
}

variables_relacionadas = []
for grupo, patron in PATRONES.items():
    for col in ess_raw.columns:
        if re.search(patron, col, flags=re.IGNORECASE):
            variables_relacionadas.append({"grupo": grupo, "variable": col})

variables_relacionadas = pd.DataFrame(variables_relacionadas).drop_duplicates()
display(variables_relacionadas)


## 9. Cobertura por ronda

Con las variables nucleares identificadas, se estudia su disponibilidad a lo largo del tiempo. Se distingue entre **valor no nulo** y **valor sustantivamente válido**: los códigos especiales de no respuesta pueden estar presentes en el fichero, pero no cuentan como información utilizable para el análisis.

Esta distinción permite detectar rondas en las que una variable existe formalmente pero no ofrece cobertura suficiente para construir el EPBI.


In [ ]:
ROUND_YEAR = {1:2002,2:2004,3:2006,4:2008,5:2010,6:2012,7:2014,8:2016,9:2018,10:2020,11:2023}

VARIABLES_COBERTURA = [v for v in [
    "hinctnta","hincfel","gndr","mnactic","eisced","eduyrs","agea",
    "lrscale","gincdif","ppltrst","trstprl","trstplt","stfdem","stfeco","stflife",
    "anweight","pspwght"
] if v in ess_raw.columns]

def mascara_valida_sustantiva(serie, variable):
    s = pd.to_numeric(serie, errors="coerce")
    if variable == "hinctnta": return s.between(1,10)
    if variable == "hincfel": return s.between(1,4)
    if variable == "gndr": return s.isin([1,2])
    if variable == "mnactic": return s.between(1,9)
    if variable == "eisced": return s.between(1,7)
    if variable == "agea": return s.between(15,123)
    if variable == "lrscale": return s.between(0,10)
    if variable == "gincdif": return s.between(1,5)
    if variable in ["ppltrst","trstprl","trstplt","stfdem","stfeco","stflife"]: return s.between(0,10)
    if variable in ["anweight","pspwght"]: return s.notna() & (s > 0)
    return serie.notna()

coverage_round=[]
for rnd,g in ess_raw.groupby("essround", dropna=False):
    for var in VARIABLES_COBERTURA:
        casos=len(g); no_nulos=int(g[var].notna().sum()); validos=int(mascara_valida_sustantiva(g[var],var).sum())
        pct_no=round(100*no_nulos/casos,2) if casos else np.nan
        pct_val=round(100*validos/casos,2) if casos else np.nan
        coverage_round.append({"essround":rnd,"survey_year":ROUND_YEAR.get(int(rnd),np.nan) if pd.notna(rnd) else np.nan,"variable":var,"casos":casos,"no_nulos":no_nulos,"pct_no_nulos":pct_no,"validos_sustantivos":validos,"pct_validos_sustantivos":pct_val,"estado":estado_cobertura(pct_val)})
coverage_round=pd.DataFrame(coverage_round)
display(coverage_round.pivot(index=["essround","survey_year"],columns="variable",values="pct_validos_sustantivos"))
alertas_round=coverage_round[coverage_round["pct_validos_sustantivos"].eq(0)].copy()
print("Variables sin valores sustantivamente válidos en alguna ronda:",len(alertas_round))
display(alertas_round)

## 10. Cobertura por país-ronda

La cobertura temporal se desagrega ahora por combinación **país-ronda**. Se calcula tanto la disponibilidad bruta como la sustantiva, y las alertas se basan en esta última.

El objetivo es pasar de una visión general por ronda a una identificación precisa de los contextos que pueden o no formar parte de la muestra analítica.


In [ ]:
if {"cntry","essround"}.issubset(ess_raw.columns):
    coverage_country_round=[]
    for (country,rnd),g in ess_raw.groupby(["cntry","essround"],dropna=False):
        for var in VARIABLES_COBERTURA:
            casos=len(g); no_nulos=int(g[var].notna().sum()); validos=int(mascara_valida_sustantiva(g[var],var).sum())
            pct_no=round(100*no_nulos/casos,2) if casos else np.nan
            pct_val=round(100*validos/casos,2) if casos else np.nan
            coverage_country_round.append({"cntry":country,"essround":rnd,"survey_year":ROUND_YEAR.get(int(rnd),np.nan) if pd.notna(rnd) else np.nan,"variable":var,"casos":casos,"no_nulos":no_nulos,"pct_no_nulos":pct_no,"validos_sustantivos":validos,"pct_validos_sustantivos":pct_val,"estado":estado_cobertura(pct_val)})
    coverage_country_round=pd.DataFrame(coverage_country_round)
    alertas_country_round=coverage_country_round[coverage_country_round["pct_validos_sustantivos"].eq(0)].copy()
    print("Combinaciones país-ronda-variable con 0% de cobertura sustantiva:",len(alertas_country_round))
    display(alertas_country_round.head(100))
else:
    coverage_country_round=pd.DataFrame(); alertas_country_round=pd.DataFrame()

## 11. Diagnóstico específico de disponibilidad del EPBI

A partir de los controles anteriores se evalúa directamente si existen observaciones potencialmente utilizables para construir el índice. Un caso es potencialmente válido cuando `hinctnta` está entre 1 y 10 y `hincfel` entre 1 y 4.

La auditoría confirma además que las rondas 1–3 quedan fuera del periodo analítico del EPBI por la ausencia de `hinctnta`. Esta decisión se aplicará de forma efectiva en el notebook 02.


In [ ]:
if {"cntry","essround","hinctnta","hincfel"}.issubset(ess_raw.columns):
    tmp=ess_raw[["cntry","essround","hinctnta","hincfel"]].copy()
    tmp["hinctnta_valido"]=mascara_valida_sustantiva(tmp["hinctnta"],"hinctnta")
    tmp["hincfel_valido"]=mascara_valida_sustantiva(tmp["hincfel"],"hincfel")
    tmp["epbi_potencial_valido"]=tmp["hinctnta_valido"] & tmp["hincfel_valido"]
    epbi_availability=(tmp.groupby(["cntry","essround"],dropna=False).agg(n_total=("cntry","size"),hinctnta_validos=("hinctnta_valido","sum"),hincfel_validos=("hincfel_valido","sum"),epbi_potencialmente_validos=("epbi_potencial_valido","sum")).reset_index())
    epbi_availability["survey_year"]=epbi_availability["essround"].map(ROUND_YEAR)
    epbi_availability["pct_hinctnta_valido"]=(100*epbi_availability["hinctnta_validos"]/epbi_availability["n_total"]).round(2)
    epbi_availability["pct_hincfel_valido"]=(100*epbi_availability["hincfel_validos"]/epbi_availability["n_total"]).round(2)
    epbi_availability["pct_epbi_potencial"]=(100*epbi_availability["epbi_potencialmente_validos"]/epbi_availability["n_total"]).round(2)
    epbi_availability["estado_epbi"]=np.select([epbi_availability["essround"].le(3),epbi_availability["epbi_potencialmente_validos"].eq(0),epbi_availability["pct_epbi_potencial"].lt(50),epbi_availability["pct_epbi_potencial"].lt(75),epbi_availability["pct_epbi_potencial"].lt(90)],["Fuera del periodo analítico EPBI (rondas 1-3)","EPBI no disponible","Cobertura EPBI muy baja","Cobertura EPBI baja","Cobertura EPBI media"],default="Cobertura EPBI alta")
    exclusion_recommendations=epbi_availability[epbi_availability["essround"].le(3) | ((epbi_availability["essround"].ge(4)) & epbi_availability["epbi_potencialmente_validos"].eq(0))].copy()
    exclusion_recommendations["motivo_exclusion"]=np.where(exclusion_recommendations["essround"].le(3),"Ronda fuera del periodo analítico EPBI: hinctnta no está disponible en las rondas 1-3","País-ronda sin observaciones sustantivamente válidas para construir el EPBI")
    epbi_no_disponible=exclusion_recommendations[(exclusion_recommendations["essround"].ge(4)) & exclusion_recommendations["epbi_potencialmente_validos"].eq(0)].copy()
    print("País-ronda posteriores a 2006 sin posibilidad de calcular EPBI:",len(epbi_no_disponible))
    display(epbi_no_disponible.sort_values(["survey_year","cntry"]))
    print("Total de combinaciones recomendadas para exclusión:",len(exclusion_recommendations))
    display(exclusion_recommendations.sort_values(["survey_year","cntry"]))
else:
    epbi_availability=pd.DataFrame(); epbi_no_disponible=pd.DataFrame(); exclusion_recommendations=pd.DataFrame()

## 12. Frecuencias de variables categóricas clave

Una vez definida la cobertura, se revisan las distribuciones de las principales variables categóricas. Las frecuencias permiten detectar códigos inesperados, categorías poco representadas y posibles valores de no respuesta que deberán tratarse durante la limpieza.


In [ ]:
VARIABLES_FRECUENCIAS = [
    v for v in ["gndr","hincfel","hinctnta","mnactic","eisced"] if v in ess_raw.columns
]

frecuencias_clave = pd.concat(
    [tabla_frecuencias(ess_raw, v, max_valores=40) for v in VARIABLES_FRECUENCIAS],
    ignore_index=True
) if VARIABLES_FRECUENCIAS else pd.DataFrame()

display(frecuencias_clave)


## 13. Valores observados e interpretación preliminar

Para complementar las frecuencias, se exportan los códigos y valores observados de las variables categóricas clave. El objetivo es facilitar su interpretación antes de recodificar o etiquetar ninguna variable.

> **Importante:** estas interpretaciones son únicamente una ayuda de auditoría. El notebook 01 no modifica los datos originales; las etiquetas y recodificaciones definitivas se aplicarán en el notebook 02.


In [ ]:
# Diccionario preliminar de interpretación para auditoría.
# No sustituye los valores originales.

INTERPRETACIONES_PRELIMINARES = {
    "gndr": {
        1: "Hombre",
        2: "Mujer",
        9: "Código especial / sin respuesta"
    },
    "hincfel": {
        1: "Vive cómodamente con los ingresos actuales",
        2: "Se las arregla con los ingresos actuales",
        3: "Le resulta difícil vivir con los ingresos actuales",
        4: "Le resulta muy difícil vivir con los ingresos actuales",
        7: "Código especial",
        8: "Código especial",
        9: "Código especial"
    },
    "hinctnta": {
        1: "Decil 1",
        2: "Decil 2",
        3: "Decil 3",
        4: "Decil 4",
        5: "Decil 5",
        6: "Decil 6",
        7: "Decil 7",
        8: "Decil 8",
        9: "Decil 9",
        10: "Decil 10",
        77: "Código especial",
        88: "Código especial",
        99: "Código especial"
    },
    "mnactic": {
        1: "Trabajo remunerado",
        2: "Educación",
        3: "Desempleado y buscando trabajo",
        4: "Desempleado y no buscando trabajo",
        5: "Enfermedad o discapacidad permanente",
        6: "Jubilado",
        7: "Servicio militar o comunitario",
        8: "Tareas del hogar o cuidados",
        9: "Otra actividad",
        77: "Código especial",
        88: "Código especial",
        99: "Código especial"
    }
}

VARIABLES_VALORES = [
    v for v in ["gndr", "hincfel", "hinctnta", "mnactic", "eisced"]
    if v in ess_raw.columns
]

filas_valores = []

for var in VARIABLES_VALORES:
    serie = ess_raw[var]
    conteos = serie.value_counts(dropna=False)

    for valor, casos in conteos.items():
        if pd.isna(valor):
            interpretacion = "Valor ausente"
            valor_export = np.nan
        else:
            valor_export = valor
            interpretacion = INTERPRETACIONES_PRELIMINARES.get(var, {}).get(
                valor,
                "Revisar codebook / sin interpretación asignada"
            )

        filas_valores.append({
            "variable": var,
            "valor_original": valor_export,
            "casos": int(casos),
            "porcentaje": round(100 * casos / len(ess_raw), 4) if len(ess_raw) else np.nan,
            "interpretacion_preliminar": interpretacion
        })

valores_variables = pd.DataFrame(filas_valores)

if not valores_variables.empty:
    valores_variables = valores_variables.sort_values(
        ["variable", "valor_original"],
        na_position="last"
    ).reset_index(drop=True)

display(valores_variables)


## 14. Valores observados por ronda

La revisión anterior se repite por ronda para comprobar si la codificación o la disponibilidad de las categorías cambia con el tiempo. Este paso ayuda a evitar que una regla de limpieza válida para una ronda se aplique de forma incorrecta a otra.


In [ ]:
filas_valores_ronda = []

if "essround" in ess_raw.columns:
    for rnd, grupo in ess_raw.groupby("essround", dropna=False):
        for var in VARIABLES_VALORES:
            conteos = grupo[var].value_counts(dropna=False)

            for valor, casos in conteos.items():
                if pd.isna(valor):
                    interpretacion = "Valor ausente"
                    valor_export = np.nan
                else:
                    valor_export = valor
                    interpretacion = INTERPRETACIONES_PRELIMINARES.get(var, {}).get(
                        valor,
                        "Revisar codebook / sin interpretación asignada"
                    )

                filas_valores_ronda.append({
                    "essround": rnd,
                    "survey_year": ROUND_YEAR.get(int(rnd), np.nan) if pd.notna(rnd) else np.nan,
                    "variable": var,
                    "valor_original": valor_export,
                    "casos": int(casos),
                    "porcentaje_dentro_ronda": round(
                        100 * casos / len(grupo), 4
                    ) if len(grupo) else np.nan,
                    "interpretacion_preliminar": interpretacion
                })

valores_por_ronda = pd.DataFrame(filas_valores_ronda)

if not valores_por_ronda.empty:
    valores_por_ronda = valores_por_ronda.sort_values(
        ["variable", "essround", "valor_original"],
        na_position="last"
    ).reset_index(drop=True)

display(valores_por_ronda.head(200))


## 15. Auditoría de códigos especiales ESS

Se identifican y contabilizan los códigos especiales de la ESS asociados a no respuesta, rechazo, desconocimiento u otras situaciones no sustantivas.

> En esta fase solo se documentan. La conversión efectiva de esos códigos a `NA` se realiza en el notebook 02.


In [ ]:
# Aquí NO se reemplaza nada. Solo se cuentan códigos que el notebook 02 deberá tratar.
SPECIAL_CODES = {
    "hinctnta": [77,88,99],
    "hincfel": [7,8,9],
    "hhmmb": [77,88,99],
    "agea": [999],
    "gndr": [9],
    "eisced": [55,77,88,99],
    "eduyrs": [77,88,99],
    "mnactic": [77,88,99],
    "lrscale": [77,88,99],
    "gincdif": [7,8,9],
    "ppltrst": [77,88,99],
    "trstprl": [77,88,99],
    "trstplt": [77,88,99],
    "stfdem": [77,88,99],
    "stfeco": [77,88,99],
    "stflife": [77,88,99],
}

special_rows = []
for var, codes in SPECIAL_CODES.items():
    if var not in ess_raw.columns:
        continue
    s_num = pd.to_numeric(ess_raw[var], errors="coerce")
    for code in codes:
        n = int(s_num.eq(code).sum())
        special_rows.append({
            "variable": var,
            "codigo_especial": code,
            "casos": n,
            "porcentaje": round(100*n/len(ess_raw), 4) if len(ess_raw) else np.nan
        })

special_codes_audit = pd.DataFrame(special_rows)
display(special_codes_audit[special_codes_audit["casos"] > 0])


## 16. Duplicados de identificación

Tras revisar el contenido de las variables, se comprueba la integridad de la identificación individual mediante la combinación `cntry + essround + idno`. Cualquier duplicado en esta clave debe quedar resuelto antes de construir una muestra analítica.


In [ ]:
duplicados_id = pd.DataFrame()

if {"cntry","essround","idno"}.issubset(ess_raw.columns):
    dup_mask = ess_raw.duplicated(["cntry","essround","idno"], keep=False)
    duplicados_id = ess_raw.loc[
        dup_mask, ["cntry","essround","idno"]
    ].sort_values(["cntry","essround","idno"])

    print("Filas implicadas en duplicados cntry-essround-idno:", len(duplicados_id))
    display(duplicados_id.head(50))


## 17. Estadísticos descriptivos

Con la estructura y las codificaciones ya auditadas, se calculan estadísticos descriptivos de las variables numéricas. Su finalidad es detectar valores extremos o patrones anómalos que puedan requerir atención en la fase de limpieza.


In [ ]:
columnas_numericas = ess_raw.select_dtypes(include=np.number).columns.tolist()
columnas_no_numericas = ess_raw.select_dtypes(exclude=np.number).columns.tolist()

descriptivos_numericos = (
    ess_raw[columnas_numericas]
    .describe()
    .transpose()
    .reset_index()
    .rename(columns={"index":"variable"})
) if columnas_numericas else pd.DataFrame()

display(descriptivos_numericos.head(40))


## 18. Visualización de valores ausentes

La información de cobertura se resume también de forma gráfica. Esta visualización permite identificar rápidamente variables o rondas con patrones de ausencia especialmente relevantes para las decisiones metodológicas del proyecto.


In [ ]:
variables_mas_missing = (
    audit_ess
    .sort_values(["porcentaje_perdidos","valores_perdidos"], ascending=False)
    .reset_index(drop=True)
)

top_missing = variables_mas_missing.head(20).sort_values("porcentaje_perdidos")

plt.figure(figsize=(10,7))
plt.barh(top_missing["variable"], top_missing["porcentaje_perdidos"])
plt.xlabel("Valores perdidos (%)")
plt.ylabel("Variable")
plt.title("ESS: 20 variables con mayor porcentaje de valores perdidos")
plt.tight_layout()

ruta_grafico_missing = GRAFICOS / "auditoria_ess_top_missing.png"
plt.savefig(ruta_grafico_missing, dpi=300, bbox_inches="tight")
plt.show()

print("Gráfico:", ruta_grafico_missing)


## 19. Exportación del informe de auditoría

Todos los diagnósticos se reúnen en un informe Excel reproducible, acompañado de un inventario de los datos brutos. Estos ficheros documentan las decisiones que se trasladarán al notebook 02; no sustituyen ni modifican los microdatos originales.


In [ ]:
ruta_informe_auditoria = INFORMES / "auditoria_inicial_ess.xlsx"
ruta_inventario = INFORMES / "inventario_datos_brutos.xlsx"

with pd.ExcelWriter(ruta_informe_auditoria, engine="openpyxl") as writer:
    resumen_ess.to_excel(writer, sheet_name="resumen_general", index=False)
    audit_ess.to_excel(writer, sheet_name="auditoria_variables", index=False)
    tipos_ess.to_excel(writer, sheet_name="tipos_datos", index=False)
    duplicados_ess.to_excel(writer, sheet_name="duplicados_completos", index=False)
    comprobacion_variables.to_excel(writer, sheet_name="variables_nucleares", index=False)
    variables_relacionadas.to_excel(writer, sheet_name="variables_relacionadas", index=False)
    coverage_round.to_excel(writer, sheet_name="cobertura_ronda", index=False)

    if not coverage_country_round.empty:
        coverage_country_round.to_excel(writer, sheet_name="cobertura_pais_ronda", index=False)
    if not alertas_country_round.empty:
        alertas_country_round.to_excel(writer, sheet_name="alertas_pais_ronda", index=False)
    if not epbi_availability.empty:
        epbi_availability.to_excel(writer, sheet_name="cobertura_EPBI", index=False)
    if not epbi_no_disponible.empty:
        epbi_no_disponible.to_excel(writer, sheet_name="EPBI_no_disponible", index=False)
    if not exclusion_recommendations.empty:
        exclusion_recommendations.to_excel(writer, sheet_name="exclusiones_recomendadas", index=False)
    if not frecuencias_clave.empty:
        frecuencias_clave.to_excel(writer, sheet_name="frecuencias_clave", index=False)
    if not valores_variables.empty:
        valores_variables.to_excel(writer, sheet_name="valores_variables", index=False)
    if not valores_por_ronda.empty:
        valores_por_ronda.to_excel(writer, sheet_name="valores_por_ronda", index=False)
    if not special_codes_audit.empty:
        special_codes_audit.to_excel(writer, sheet_name="codigos_especiales", index=False)
    if not duplicados_id.empty:
        duplicados_id.to_excel(writer, sheet_name="duplicados_id", index=False)
    if not descriptivos_numericos.empty:
        descriptivos_numericos.to_excel(writer, sheet_name="descriptivos_numericos", index=False)

inventario_brutos.to_excel(ruta_inventario, index=False)

print("Auditoría exportada correctamente.")
print("Informe:", ruta_informe_auditoria)
print("Inventario:", ruta_inventario)


## 20. Cierre de la fase 01 y paso al notebook 02

La auditoría termina cuando existe evidencia suficiente para fijar las reglas de limpieza. En particular, deben quedar respondidas estas preguntas:

1. ¿Están presentes todas las variables nucleares?
2. ¿Qué variables están completamente ausentes por ronda?
3. ¿Qué combinaciones país-ronda tienen cobertura sustantiva nula o muy baja?
4. ¿En qué países-ronda no puede calcularse el EPBI con `hinctnta` entre 1–10 y `hincfel` entre 1–4?
5. ¿Qué rondas quedan fuera del periodo analítico del EPBI?
6. ¿Qué códigos aparecen realmente en las variables categóricas clave?
7. ¿Qué códigos especiales deben convertirse a `NA`?
8. ¿Qué combinaciones país-ronda deben excluirse por ausencia total de inputs válidos?
9. ¿Existen duplicados en `cntry + essround + idno`?

### Decisiones que se trasladan a la limpieza

- Los datos brutos permanecen intactos.
- La muestra analítica del EPBI comienza en la **ronda 4 (2008)**; las rondas 1–3 se excluirán en el notebook 02 porque `hinctnta` no está disponible.
- Desde la ronda 4, cualquier país-ronda con **0 observaciones potencialmente válidas para EPBI** se excluirá de la muestra analítica.
- Los códigos especiales, incluidos `77/88/99` en `hinctnta`, se convertirán a `NA` en la fase de limpieza.
- Las decisiones se aplicarán sobre una copia de trabajo, preservando siempre el fichero ESS original.

Con estas reglas cerradas, el **notebook 02** transforma la ESS bruta en una muestra limpia y elegible para la construcción posterior del EPBI.
